# Text-to-Audio

> Generating non-speech audio - music and sound effects - from a text prompt: how the models work, the mid-2026 open landscape, evaluation, and runnable transformers / diffusers code on GPU or CPU.

- skip_showdoc: true
- skip_exec: true

## 1. What is Text-to-Audio?

Text-to-Audio (TTA) generates **general audio** - music, sound effects, ambiences - from a natural-language prompt. It is distinct from Text-to-Speech (which generates spoken words); the two are usually separate models.

**Input.** A text prompt ("lo-fi hip hop with a mellow piano", "rain on a tin roof, distant thunder"), optionally with a **duration**, a **melody** to condition on, or a seed.

**Output.** A mono or stereo waveform, typically **16 kHz** (MusicGen) up to **44.1 kHz stereo** (Stable Audio Open).

**Neighbouring tasks:**

| Task | What it does | Typical tool |
|------|--------------|--------------|
| Text-to-Speech | Generate spoken words | SpeechT5, Bark |
| Audio captioning | The inverse: describe a sound | Qwen2-Audio, CLAP |
| Music continuation | Extend an audio clip | MusicGen (melody) |
| Audio-to-audio | Transform existing audio | Demucs, enhancement |

---

## 2. How Modern Text-to-Audio Works

Two dominant families, converging on quality:

1. **Autoregressive codec LM (AudioGen, MusicGen, 2023).** Encode audio into discrete **EnCodec** tokens, then predict them with a transformer LM conditioned on **T5** text embeddings. Simple, streamable, strong on music; Meta's MusicGen is the open reference.
2. **Latent diffusion (AudioLDM, AudioLDM 2, Tango, 2023).** Run a diffusion model in a compressed VAE latent, conditioned on **CLAP** and/or T5 text features, then decode + vocode to waveform. Strong on general sound effects.
3. **Latent diffusion transformers / flow (Stable Audio, Stable Audio Open, 2024).** A DiT with **timing conditioning** generates long, high-fidelity **44.1 kHz stereo** in one pass. State of the art for open SFX and loops.
4. **2025-2026.** Faster sampling (consistency / flow-matching), longer coherent structure, and strong closed systems (Suno, Udio) alongside open MusicGen / Stable Audio Open / AudioLDM 2.

---

## 3. Evaluation Metrics

Generation has no single reference output, so metrics compare **distributions** and **prompt adherence**.

- **FAD (Frechet Audio Distance).** Distance between the feature distributions (VGGish / PANN embeddings) of generated vs real audio. Lower = more realistic. The primary TTA metric.
- **KL divergence.** Between PANN class-probability distributions of generated vs reference - measures semantic match.
- **CLAP score.** Cosine similarity between the **CLAP** text embedding of the prompt and the audio embedding of the output - measures how well the audio matches the prompt (higher is better).
- **IS (Inception Score)** and subjective **MOS** for quality/diversity.

The cell below computes a CLAP score with `transformers` - the one metric you can run on a single clip.

---

In [ ]:
import numpy as np
import torch
from transformers import ClapModel, ClapProcessor

clap_id = "laion/clap-htsat-unfused"
clap = ClapModel.from_pretrained(clap_id)
clap_proc = ClapProcessor.from_pretrained(clap_id)


def clap_score(prompt, audio, sr):
    "Cosine similarity between the prompt and the audio in CLAP space (prompt adherence)."
    if sr != 48000:  # CLAP expects 48 kHz
        import librosa
        audio = librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=sr, target_sr=48000)
    inp = clap_proc(text=[prompt], audio=[audio], sampling_rate=48000, return_tensors="pt", padding=True)
    with torch.no_grad():
        emb = clap(**inp)
    a = torch.nn.functional.normalize(emb.audio_embeds, dim=-1)
    t = torch.nn.functional.normalize(emb.text_embeds, dim=-1)
    return float((a * t).sum())


# Sanity check on random noise (should score low)
print("noise CLAP score:", clap_score("a solo piano melody", np.random.randn(48000) * 0.01, 48000))

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

noise CLAP score: 0.08860747516155243


## 4. The Model Landscape (mid-2026)

| Model | Params | License | Type | Library | Best for |
|-------|--------|---------|------|---------|----------|
| facebook/musicgen-small | 300M | CC-BY-NC 4.0 | music (codec LM) | transformers | music from text, melody conditioning |
| facebook/musicgen-stereo-small | 300M | CC-BY-NC 4.0 | stereo music | transformers | stereo music |
| cvssp/audioldm-s-full-v2 | 350M | CC-BY-NC | sound + music (LDM) | diffusers | general sound effects |
| cvssp/audioldm2 | 350M | CC-BY-NC | sound + music (LDM) | diffusers | successor (see note) |
| stabilityai/stable-audio-open-1.0 | 1.1B | Stability Community | 44.1 kHz stereo (DiT) | diffusers | high-fidelity SFX and loops |
| facebook/audiogen-medium | 1.5B | CC-BY-NC 4.0 | sound effects | audiocraft (external) | dense sound scenes |

There is no single crowd leaderboard; papers report FAD/KL/CLAP on **AudioCaps** (sound) and **MusicCaps** (music). MusicGen loads through `transformers`; AudioLDM and Stable Audio Open load through **`diffusers`** (the standard Hugging Face diffusion library - a general-purpose lib, so it fits the repo rule). AudioGen needs the external `audiocraft` runtime.

We run **AudioLDM v1** (`AudioLDMPipeline`) below: it conditions on the CLAP text encoder directly, so it stays robust across `transformers` versions. The newer **AudioLDM 2** uses an internal GPT-2 language-model prompt path that currently breaks against `transformers` 5.x inside `diffusers` - use it only with an older, matched `diffusers`/`transformers` pair.

---

## 5. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - MusicGen and CLAP
- `diffusers` - AudioLDM (and Stable Audio Open)
- `soundfile` - write WAV; `librosa` - resample for CLAP
- `pyecharts` - the benchmark chart

MusicGen and CLAP are already covered by the repo deps; `diffusers` is the one extra general-purpose library used here.

---

In [ ]:
import gc
import time
import urllib.request
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:16s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "GC then release cached CPU/GPU memory. Call right after `del model`.\n\n    `del` drops the Python reference; this reclaims the RAM and hands the\n    freed VRAM back to the CUDA allocator so usage stays flat across cells.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# All downloads (samples, HF cache) go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)

import soundfile as sf

PROMPT = "a warm lo-fi hip hop beat with a mellow piano and soft vinyl crackle"
OUT_DIR = DATA_DIR / "tta_out"
OUT_DIR.mkdir(exist_ok=True)


def save_wav(name, audio, sr):
    path = OUT_DIR / name
    sf.write(path, np.asarray(audio).squeeze(), sr)
    print(f"{name}: {np.asarray(audio).squeeze().shape[-1] / sr:.1f} s @ {sr} Hz -> {path}")
    return path

NVIDIA GeForce RTX 3060
device: cuda:0


## 6. MusicGen (transformers)

The `text-to-audio` pipeline wraps MusicGen end to end. `max_new_tokens` sets the length (~50 tokens/second of audio); the output sampling rate comes from the model config (32 kHz). Use `facebook/musicgen-melody` to additionally condition on a hummed melody.

---

In [ ]:
from transformers import pipeline

musicgen = pipeline("text-to-audio", model="facebook/musicgen-small", device=device)
t0 = time.perf_counter()
out = musicgen(PROMPT, forward_params={"do_sample": True, "max_new_tokens": 256})  # ~5 s of audio
print(f"{time.perf_counter() - t0:.1f}s")
save_wav("musicgen.wav", out["audio"], out["sampling_rate"])

del musicgen
free_memory()
vram("after musicgen")

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


2.9s
musicgen.wav: 5.1 s @ 32000 Hz -> ../../datasets/tta_out/musicgen.wav
VRAM after musicgen    0.01 GB allocated /  0.02 GB reserved


## 7. AudioLDM (diffusers)

AudioLDM is a latent-diffusion model good at general **sound effects** as well as music. It loads through `diffusers` and conditions on a CLAP text encoder; `num_inference_steps` trades speed for quality and `audio_length_in_s` sets duration. Runs on CPU (slowly) or GPU with `torch_dtype=float16`, and outputs 16 kHz.

---

In [ ]:
from diffusers import AudioLDMPipeline

dtype = torch.float16 if device != "cpu" else torch.float32
ldm = AudioLDMPipeline.from_pretrained("cvssp/audioldm-s-full-v2", torch_dtype=dtype,
                                       cache_dir=str(DATA_DIR / "hf_cache")).to(device)

t0 = time.perf_counter()
audio = ldm("rain on a tin roof with distant thunder", num_inference_steps=50,
            audio_length_in_s=5.0).audios[0]
print(f"{time.perf_counter() - t0:.1f}s")
save_wav("audioldm.wav", audio, 16000)  # AudioLDM outputs 16 kHz

del ldm
free_memory()
vram("after audioldm")

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


  0%|          | 0/50 [00:00<?, ?it/s]

1.2s
audioldm.wav: 5.0 s @ 16000 Hz -> ../../datasets/tta_out/audioldm.wav
VRAM after audioldm    0.01 GB allocated /  0.02 GB reserved


## 8. Head-to-head Benchmark

Same prompt through MusicGen and AudioLDM; report **generation RTF** and **CLAP score** (prompt adherence). CLAP is an automatic proxy - for real evaluation use FAD/KL over AudioCaps/MusicCaps. Free each model before loading the next.

---

In [ ]:
# ECharts (pyecharts) is the repo standard for all charts - it renders interactive
# and embeds straight into the Quarto docs via .render_notebook().
from pyecharts import options as opts
from pyecharts.charts import Bar


def bar_chart(title, categories, series, y_name=""):
    "Grouped bar chart. `series` is a dict {name: [values aligned to categories]}."
    chart = Bar(init_opts=opts.InitOpts(width="720px", height="420px"))
    chart.add_xaxis([str(c) for c in categories])
    for name, vals in series.items():
        chart.add_yaxis(name, [round(float(v), 4) for v in vals])
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title=title),
        yaxis_opts=opts.AxisOpts(name=y_name),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=20)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
    return chart.render_notebook()

In [ ]:
from transformers import pipeline
from diffusers import AudioLDMPipeline

results = {}

# MusicGen
mg = pipeline("text-to-audio", model="facebook/musicgen-small", device=device)
t0 = time.perf_counter()
out = mg(PROMPT, forward_params={"do_sample": True, "max_new_tokens": 256})
gen_s = time.perf_counter() - t0
audio, sr = np.asarray(out["audio"]).squeeze(), out["sampling_rate"]
results["musicgen"] = {"rtf": gen_s / (len(audio) / sr), "clap": clap_score(PROMPT, audio, sr)}
del mg
free_memory()

# AudioLDM
dtype = torch.float16 if device != "cpu" else torch.float32
ldm = AudioLDMPipeline.from_pretrained("cvssp/audioldm-s-full-v2", torch_dtype=dtype,
                                       cache_dir=str(DATA_DIR / "hf_cache")).to(device)
t0 = time.perf_counter()
audio = np.asarray(ldm(PROMPT, num_inference_steps=50, audio_length_in_s=5.0).audios[0])
gen_s = time.perf_counter() - t0
results["audioldm"] = {"rtf": gen_s / (len(audio) / 16000), "clap": clap_score(PROMPT, audio, 16000)}
del ldm
free_memory()

for name, r in results.items():
    print(f"{name:12s} RTF {r['rtf']:6.2f}  CLAP {r['clap']:.3f}")
vram("after benchmark")

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


  0%|          | 0/50 [00:00<?, ?it/s]

musicgen     RTF   0.46  CLAP 0.628
audioldm     RTF   0.19  CLAP 0.211
VRAM after benchmark   0.01 GB allocated /  0.02 GB reserved


In [ ]:
names = list(results)
bar_chart(
    "Text-to-Audio: prompt adherence (CLAP, higher better) and RTF",
    names,
    {"CLAP score": [results[n]["clap"] for n in names],
     "RTF": [results[n]["rtf"] for n in names]},
    y_name="score",
)

## 9. Going Further

- **Higher fidelity.** `stabilityai/stable-audio-open-1.0` (diffusers `StableAudioPipeline`, gated) generates 44.1 kHz stereo up to ~47 s - the best open SFX/loop quality.
- **Melody conditioning.** `facebook/musicgen-melody` continues or re-harmonizes a reference melody.
- **Longer / structured music.** MusicGen supports sliding-window continuation; commercial Suno / Udio lead on song structure.
- **Sound effects (external).** `facebook/audiogen-medium` via `audiocraft` specializes in dense sound scenes.
- **Evaluation at scale.** Use the `audioldm_eval` toolkit for FAD/KL/IS over AudioCaps / MusicCaps.

---